In [1]:
#@title Setup (~13mins)
%%capture
structure_scoring = "Boltz"  # @param ['Boltz', 'Boltz_template', 'ProtenixScore']

if structure_scoring == "ProtenixScore":
  !git clone https://github.com/cytokineking/ProtenixScore
  %cd ProtenixScore
  !./install_protenixscore.sh --no-conda
  %cd /content
  !pip install https://data.pyg.org/whl/torch-2.7.0%2Bcu126/torch_cluster-1.6.3%2Bpt27cu126-cp312-cp312-linux_x86_64.whl
if structure_scoring == "Boltz" or structure_scoring == "Boltz_template":
  !pip install virtualenv
  !virtualenv boltz_env
  !source /content/boltz_env/bin/activate; pip install boltz[cuda] -U
  !pip uninstall torch torchvision torchaudio -y
  !pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
  !pip install https://data.pyg.org/whl/torch-2.8.0%2Bcu128/torch_cluster-1.6.3%2Bpt28cu128-cp312-cp312-linux_x86_64.whl
  !pip install lightning[extra]

!pip install torch_geometric
!pip install pyrosetta-installer
!python -c 'import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()'
!pip install biopython
!pip install tqdm
!pip install dm-tree
!pip install biotite
!pip install pyyaml
!pip install modelcif
!pip install e3nn
#!pip install easydict


!git clone https://github.com/nedru004/MPNN_affinity_maturation.git
#working_dir = "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation"
working_dir = "/content/MPNN_affinity_maturation"

!git clone https://github.com/dauparas/ProteinMPNN.git

!git clone https://gitlab.com/mjslee0921/flowpacker
#%pip install --use-deprecated=legacy-resolver -r flowpacker/requirements.txt

#!git clone https://github.com/ohuelab/boltzina.git
#!https://github.com/nedru004/boltz_score.git

import sys
sys.path.insert(0, working_dir)
sys.path.insert(0, "/content/flowpacker")

In [2]:
#@title 1. MPNN Squence Design (~15s)
import subprocess
import sys
import json # Added for JSON handling
import os   # Added for path operations

mpnn_script_path = f"ProteinMPNN/protein_mpnn_run.py" # Renamed 'script' to avoid conflict
folder_with_pdbs = f"{working_dir}/test/"
path_for_parsed_chains = f"{working_dir}/test/parsed_chains.jsonl"
path_for_assigned_chains = f"{working_dir}/test/assigned_chains.jsonl"
path_for_fixed_positions = f"{working_dir}/test/fixed_positions.jsonl" # Ensuring it is jsonl as per MPNN and code definition
path_for_fixed_positions_csv = f"{working_dir}/test/fixed_positions.csv"
out_dir = f"{working_dir}/test/mpnn_out/"
out_dir_bias = f"{working_dir}/test/mpnn_out_bias/"
bias_AA_file = f"{working_dir}/bias_AA.jsonl"
chains_to_design ="A" # @param {type: "string"}
num_seq_per_target=8  # @param {type: "number"}
sampling_temp =0.5  # @param {type: "number"}
seed = 37  # @param {type: "number"}
fixed_positions="" # @param {type: "string"}
#generate fixed position csv
with open(path_for_fixed_positions_csv, 'w') as f:
  f.write(fixed_positions)

!python ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path=$folder_with_pdbs --output_path=$path_for_parsed_chains

!python ProteinMPNN/helper_scripts/assign_fixed_chains.py --input_path=$path_for_parsed_chains --output_path=$path_for_assigned_chains --chain_list "$chains_to_design"

!python ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path=$path_for_parsed_chains --output_path=$path_for_fixed_positions --chain_list "$chains_to_design" --position_list "$fixed_positions"

# Run actual protein_mpnn_run script
args_for_first_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--use_soluble",
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C"]

subprocess.run(args_for_first_mpnn_run)


# MPNN with AA bias
# Ensure correct script path and flag for fixed positions.
args_for_second_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--use_soluble",
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir_bias,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C",
    "--bias_AA_jsonl", bias_AA_file]

subprocess.run(args_for_second_mpnn_run)

CompletedProcess(args=['/usr/bin/python3', 'ProteinMPNN/protein_mpnn_run.py', '--use_soluble', '--jsonl_path', '/content/MPNN_affinity_maturation/test/parsed_chains.jsonl', '--chain_id_jsonl', '/content/MPNN_affinity_maturation/test/assigned_chains.jsonl', '--fixed_positions_jsonl', '/content/MPNN_affinity_maturation/test/fixed_positions.jsonl', '--out_folder', '/content/MPNN_affinity_maturation/test/mpnn_out_bias/', '--num_seq_per_target', '8', '--sampling_temp', '0.5', '--seed', '37', '--batch_size', '8', '--omit_AAs', 'C', '--bias_AA_jsonl', '/content/MPNN_affinity_maturation/bias_AA.jsonl'], returncode=0)

In [3]:
#@title 2. Run flowpacker (~30s)

from utilities import direct_fasta_to_csv

mpnn_folders = [f"{working_dir}/test/mpnn_out/seqs", f"{working_dir}/test/mpnn_out_bias/seqs"]
direct_fasta_to_csv(input_dirs=mpnn_folders, output_csv=f"{working_dir}/test/mpnn_final_result.csv", suffix=".pdb")

# Edit import in flowpacker
file_path = '/content/flowpacker/utils/loader.py'
line_number_to_edit = 45 # Line numbers are 1-based for users
new_content = "    config_dir = '/content/MPNN_affinity_maturation/base.yaml'"
with open(file_path, 'r') as file:
    lines = file.readlines()
# Adjust for 0-based indexing (line 3 is at index 2)
if 0 < line_number_to_edit <= len(lines):
    lines[line_number_to_edit - 1] = new_content + '\n' # Ensure a newline character is present
# Step 3: Write the modified list back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)


# Run flowpacker
!python $working_dir/sampler_pdb_colab.py base --use_gt_masks True --pdb_dir $working_dir/test/ --save_dir $working_dir/test/flowpacker_out/ --csv_file $working_dir/test/mpnn_final_result.csv

✅ Processing complete! 16 unique sequences have been written to: /content/MPNN_affinity_maturation/test/mpnn_final_result.csv
/content/flowpacker/models/equiformer_v2/layer_norm.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/flowpacker/models/equiformer_v2/layer_norm.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/flowpacker/models/equiformer_v2/layer_norm.py:227: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/content/flowpacker/models/equiformer_v2/layer_norm.py:310: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp

In [4]:
#@title 3. Score and filter binders for pTM and ipTM (Boltz on T4 26m - 15m setup)
from MPNN_affinity_maturation.utilities import filter_protenix_scores, filter_boltz_scores, generate_boltz_yamls_from_pdbs

if structure_scoring == "ProtenixScore":
  # Edit import
  file_path = '/content/ProtenixScore/score.py'
  with open(file_path, 'r') as file:
      lines = file.readlines()
  # Adjust for 0-based indexing (line 3 is at index 2)
  lines[19 - 1] = 'import sys\nsys.path.insert(0, "/content/ProtenixScore/Protenix_fork")\n' # Ensure a newline character is present
  lines[1070 - 1] = '    from ProtenixScore.msa_colabfold import ColabFoldMSAConfig, ensure_msa_dir\n'
  # Step 3: Write the modified list back to the file
  with open(file_path, 'w') as file:
      file.writelines(lines)
  # Edit import
  file_path = '/content/ProtenixScore/cli.py'
  new_content = 'from ProtenixScore.score import run_score'
  with open(file_path, 'r') as file:
      lines = file.readlines()
  lines[5 - 1] = new_content + '\n' # Ensure a newline character is present
  # Step 3: Write the modified list back to the file
  with open(file_path, 'w') as file:
      file.writelines(lines)
  !python ProtenixScore/cli.py score --input $working_dir/test/flowpacker_out/after_pdbs/ --output $working_dir/test/protenix_scores/ --recursive
  # Filter scores
  filter_protenix_scores(f"{working_dir}/test/protenix_scores/", f"{working_dir}/test/structure_score_summary.csv", f"{working_dir}/test/filtered_pdb/", f"{working_dir}/test/flowpacker_out/after_pdbs/", threshold_pTM = 0.8, threshold_ipTM = 0.8)

if structure_scoring == "Boltz" or structure_scoring == "Boltz_template":
  # Boltz full prediction
  # write a yaml file for each pdb
  use_template = structure_scoring == "Boltz_template"
  generate_boltz_yamls_from_pdbs(f"{working_dir}/test/flowpacker_out/after_pdbs/", f"{working_dir}/test/boltz_input/", use_template = use_template)
  for yaml in os.listdir(f"{working_dir}/test/boltz_input/"):
    if yaml.endswith(".yaml"):
      !source /content/boltz_env/bin/activate; boltz predict $working_dir/test/boltz_input/$yaml --out_dir $working_dir/test/boltz_out/ --use_msa_server
  # calculate score and RMSD
  filter_boltz_scores(f"{working_dir}/test/boltz_out/", f"{working_dir}/test/structure_score_summary.csv", f"{working_dir}/test/filtered_pdb/", f"{working_dir}/test/flowpacker_out/after_pdbs/", threshold_pTM = 0.8, threshold_ipTM = 0.8, threshold_rmsd = 3)


Generating Boltz YAMLs: 100%|██████████| 8/8 [00:00<00:00, 276.74it/s]


✅ Generated 8 Boltz YAML files in /content/MPNN_affinity_maturation/test/boltz_input/
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Extracting the CCD data to /root/.boltz/mols. This may take a bit of time. You may change the cache directory with the --cache flag.
Checking input data.
Processing 1 inputs with 1 threads.
  0% 0/1 [00:00<?, ?it/s]Generating MSA for /content/MPNN_affinity_maturation/test/boltz_input/EPO12_V3-12_8.yaml with 2 protein entities.
Calling MSA server for target EPO12_V3-12_8 with 2 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server

  0% 0/300 [00:00<?, ?it/s]
SUBMIT:   0% 0/300 [00:00<?, ?it/s]
COMPLETE:   0% 0/300 [00:00<?, ?it/s]
COMPLETE: 100% 300/300 [00:01<00:00, 231.56it/s]

  0% 0/300 [00:00<?, ?it/s]
SUBMIT:   0% 0/300 [00:00<?, ?it/s]
COMPLETE:   0% 0/300 [00:00<?, ?it/s]
COMPLETE: 100% 300/300 [00:01<00:00, 238.66it/s]
100% 1/1 [

Parsing JSON files: 100%|██████████| 24/24 [00:02<00:00, 11.50it/s]

✅ Extracted scores from 8 files to /content/MPNN_affinity_maturation/test/structure_score_summary.csv
   3 PDB(s) passed thresholds (pTM ≥ 0.8, ipTM ≥ 0.8, RMSD ≤ 3)
   RMSD computed for 8/8 entries


In [ ]:
#@title 4. Rosetta interaction energy (16 mins for 8 structures)
from Bio import SeqIO
import os

#pdb_folder = "test"
pdb_folder = f"{working_dir}/test/filtered_pdb"
interface_dist = 10
energy_filter = -5

# find pdbs in folder
pdb_files = []
for file in os.listdir(pdb_folder):
    if file.endswith(".pdb"):
        pdb_files.append(file)
        print(file)

# update xml file
# open pdb to extract residues
for pdb in pdb_files:
  # extract amino acids
  with open(os.path.join(pdb_folder,pdb), 'r') as pdb_file:
    resnums = "" # should include all positions "1A-119A,1M-61M"
    for record in SeqIO.parse(pdb_file, 'pdb-atom'):
      resnums += "1" + record.id.split(":")[1] + "-" + str(len(record.seq)) + record.id.split(":")[1] + ","

  with open(f"{working_dir}/update.xml", "r") as f:
      xml = f.readlines()
      # edit line 5 resnums
      xml[4] = f'\t\t<Neighborhood name="nbrhood" resnums="{resnums[:-1]}" distance="20.0"/>\n'
      xml[5] = f'\t\t<!-- <Neighborhood name="nbrhood" resnums="{resnums[:-1]}" distance="20.0"/> -->\n'
      # write xml
      with open(f"{working_dir}/edited.xml", "w") as f:
          f.writelines(xml)

  # run energy calculation
  !python $working_dir/per_residue_energy_pyrosetta.py --pdb $pdb_folder/$pdb --binder_id A --target_id M --interface_dist $interface_dist --output_dir $working_dir/test/ --xml_protocol $working_dir/edited.xml

EPO12_V3-12_2.pdb
EPO12_V3-12_8.pdb
EPO12_V3-12_4.pdb


/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 5038
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/SeqIO/PdbIO.py:337: BiopythonParserWarning: 'HEADER' line not found; can't determine PDB ID.
  warnings.warn(


┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python312.ubuntu 2026.06+release.e5a76a2dbd7e42e1271d0af460926c86e29e59d1 2026-02-02T18:21:37] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags 
core.init: Rosetta version: PyRosetta4.Release.python312.ubuntu r425 2026.06+release.e5a76a2dbd e

In [41]:
#@title 5. Concatenate key residues

from utilities import process_pdb_mutation_and_renumber
# merge key residues into one pdb file

# Extract Sequences
#!python demo_scripts/pdb2fa.py $working_dir/test/af3score/filtered_links csv $working_dir/test/af3score/filtered_links.csv

process_pdb_mutation_and_renumber(f"{working_dir}/test/fixed_residue_pyrosetta.csv", f"{working_dir}/test/merge_motif_pdb/", renumber_chain='M', start_index=9) # The residue numbering of the target chain is consistent with the input target file

FileNotFoundError: [Errno 2] No such file or directory: '/content/MPNN_affinity_maturation/test/fixed_residue_pyrosetta.csv'

In [ ]:
#@title 6. Partial Diffusion on non-fixed residues

#run rfdiffusion
partial_T = "20" # @param ["auto", "10", "20", "40", "60", "80"]
num_designs = 1 # @param {type: "number"}

print("installing RFdiffusion...")
os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
# 17Mar2024: adding --no-dependencies to avoid installing nvidia-cuda-* dependencies
# 25Aug2025: updating dgi install to work with latest pytorch
os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
os.system("wget -qnc https://files.ipd.uw.edu/krypton/ananas")
os.system("chmod +x ananas")

print("downloading RFdiffusion params...")
os.system("mkdir RFdiffusion/models")
models = ["Base_ckpt.pt","Complex_base_ckpt.pt","Complex_beta_ckpt.pt"]
for m in models:
  while os.path.isfile(f"{m}.aria2"):
    time.sleep(5)
os.system(f"mv {' '.join(models)} RFdiffusion/models")
os.system("unzip schedules.zip; rm schedules.zip")

sys.path.append('RFdiffusion')

diffuse_target = f"{working_dir}/test/merge_motif_pdb/"
contigs = ""  # We could instead just noise the latter 59 residues, by specifying [A1-19/59]

!python RFdiffusion/run_inference.py inference.output_prefix=MPNN_affinity_maturation/test/partial_diffusion/test inference.input_pdb=input_pdbs/$diffuse_target inference.num_designs=$partial_T diffuser.T=50 'contigmap.contigs=[$contigs]'